In [ ]:
import json
import warnings
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")

### Training rolling window Doc2Vec models and computing term similarities

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────
DATA_PATH  = "all_bundestag_speeches_preprocessed.csv"
OUTPUT_DIR = Path("output_dictionaries")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── YOUR COLUMN NAMES ────────────────────────────────────────
TEXT_COL      = "text_preprocessed_lemmatized"  # already lemmatised ✓
FALLBACK_COL  = "text_preprocessed"             # if lemmatized is empty
DATE_COL      = "date_year"
PARTY_COL     = "Party"
SEP           = ";"                            # tab-separated


df = pd.read_csv(DATA_PATH, sep=SEP, low_memory=False)

In [ ]:
def find_terms_vectors(terms_list, model, vocabulary):
    '''
    :param terms_list: a list of relevant terms embeddings of which should be found in the model.
    :param model: a Doc2Vec gensim model.
    :param vocabulary: vocabulary of the model.
    :return: a list of terms found in the vocabulary and a list of corresponding vectors.
    '''
    wv = []
    terms = []
    for i in terms_list:
        if len(i.split()) == 1:
            if i in vocabulary:
                wv.append(model.wv[i])
                terms.append(i)
        elif len(i.split()) == 2:
            if (i.split()[0] in vocabulary) & (i.split()[1] in vocabulary):
                wv.append(model.wv[i.split()[0]]+model.wv[i.split()[1]])
                terms.append(i)
    return terms, wv

In [ ]:
import os
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import pandas as pd
import multiprocessing
import time
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path
# Get the number of CPU cores for multiprocessing
cores = multiprocessing.cpu_count()

# Import data
data = df
data.index = pd.PeriodIndex(data.date, freq='Q')

# Create a list of periods and sub-periods for the analysis
periods = pd.PeriodIndex(pd.date_range('1960-01-01', periods=262, freq='Q'))
subperiods = []
forecast_periods = []
for i in range(0, 222, 1):
    subperiods.append(periods[i:i + 40])
    forecast_periods.append(periods[i + 40])

# Add columns for similarity metrics to the data frame
data['similarity_expansive'] = np.nan
data['similarity_restrictive'] = np.nan

# Set parameters for the Doc2Vec model
set_window = 5
set_epochs = 10
set_vector_size = 100
dictionary = 'extended'

# Define the directory to save the trained models
models_directory = Path(f"results/doc2vec_models/{set_window}Window_{set_epochs}Epochs_{set_vector_size}Size")
models_directory.mkdir(parents=True, exist_ok=True)

# Load fiscal policy terms
expansive = pd.read_csv('dictionaries/welcoming_terms_preprocessed.csv')['welcoming_terms'].values
restrictive = pd.read_csv('dictionaries/restrictive_terms_preprocessed.csv')['restrictive_terms'].values

# Track the total start time
starttime = time.time()

# Loop through each subperiod
for i, period in enumerate(subperiods):
    print(f"Train model for the period {period[0]}-{period[-1]}.")
    start = time.time()

    # Train model based on ten years of data
    subset = data[data.index.isin(list(period))].reset_index(drop=True)
    sentences = subset.text_preprocessed_lemmatized
    sentences = [i.split() for i in sentences]
    documents = [TaggedDocument(doc, [id_]) for id_, doc in enumerate(sentences)]

    # Create and train the Doc2Vec model
    model = Doc2Vec(documents,
                    workers=cores - 1,
                    alpha=0.025,
                    vector_size=set_vector_size,
                    min_count=1,
                    epochs=set_epochs,
                    window=set_window,
                    dbow_words=1)
    print(f'The estimation took {time.time() - start} seconds.')

    # Save the trained model
    model.save(f'{models_directory}/doc2vec_subset_{period[0]}_{period[-1]}')

    print(
        f'Calculate similarities to migration policy vectors for the period {forecast_periods[i]}.')

    # Calculate "welcoming" and "restrictive" vectors for the current period
    vocabulary = [word for index, word in enumerate(model.wv.index_to_key)]
    expansive_terms, expansive_wv = find_terms_vectors(expansive, model, vocabulary)
    restrictive_terms, restrictive_wv = find_terms_vectors(restrictive, model, vocabulary)
    vector_expansive = sum(expansive_wv) / len(expansive_wv)
    vector_restrictive = sum(restrictive_wv) / len(restrictive_wv)

    # Forecast for the next quarter
    forecast_df = data[data.index == forecast_periods[i]]
    document_vectors = [model.infer_vector(text.split()) for text in forecast_df.text_preprocessed_lemmatized]

    # Calculate similarities and store them in the DataFrame
    data.loc[forecast_periods[i], 'similarity_expansive'] = [cosine_similarity([vec], [vector_expansive])[0][0] for vec
                                                             in document_vectors]
    data.loc[forecast_periods[i], 'similarity_restrictive'] = [cosine_similarity([vec], [vector_restrictive])[0][0] for
                                                               vec in document_vectors]

# Track the total end time
endtime = time.time()
print(f'Overall the process took {(endtime - starttime) / 60} minutes.')

# Save the DataFrame to a CSV file
data.to_csv(f'results/doc2vec_10Years_RollingWindow.csv', index=False)

### Computing net migration sentiment: Government vs. opposition (quarterly aggregation)

In [ ]:
data = pd.read_csv("results/doc2vec_10Years_RollingWindow.csv", low_memory=False)

# make sure sentiment is numeric
data['sentiment'] = pd.to_numeric(
    data['similarity_expansive'], errors='coerce'
) - pd.to_numeric(
    data['similarity_restrictive'], errors='coerce'
)

# drop broken rows
data = data.dropna(subset=['sentiment'])

# group ONLY sentiment column
sentiment_full = data.groupby(['date_year', 'date_quarter'])['sentiment'].mean()

sentiment_government = data[data.governing_Party == 1] \
    .groupby(['date_year', 'date_quarter'])['sentiment'].mean()

sentiment_opposition = data[data.governing_Party == 0] \
    .groupby(['date_year', 'date_quarter'])['sentiment'].mean()

data_quarterly = pd.concat([
    sentiment_full.rename("sentiment_gesamt"),
    sentiment_government.rename("sentiment_government"),
    sentiment_opposition.rename("sentiment_opposition")
], axis=1)

data_quarterly.to_csv("results/fiscal_sentiment_quarterly.csv")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ── LOAD DATA ────────────────────────────────────────────────
df = pd.read_csv("results/fiscal_sentiment_quarterly.csv")

# Create proper datetime index from Year + Quarter
df["date"] = pd.PeriodIndex(
    year=df["date_year"],
    quarter=df["date_quarter"],
    freq="Q"
).to_timestamp()

df = df.sort_values("date")
df.set_index("date", inplace=True)

# ── PLOT ─────────────────────────────────────────────────────
plt.figure(figsize=(12, 6))

plt.plot(df.index, df["sentiment_gesamt"], label="Total")
plt.plot(df.index, df["sentiment_government"], label="Government")
plt.plot(df.index, df["sentiment_opposition"], label="Opposition")

# zero line (important for interpretation)
plt.axhline(0)

plt.title("Migration Sentiment in Bundestag (Quarterly)")
plt.xlabel("Year")
plt.ylabel("Sentiment (Expansive − Restrictive)")
plt.legend()

plt.tight_layout()
plt.show()